# Load Test Data Visualization (Seaborn)

横轴: 真实重量(kg)
纵轴: 估计的 Z 轴力(N, `ext_wrench[2]`)

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import sys

sys.path.insert(0, str(Path.cwd().parent))
from tests.figure_export import save_publication_figure

sns.set_theme(style="whitegrid")

def load_data(path):
    data_path = Path(path)
    with data_path.open("r", encoding="utf-8") as f:
        raw = json.load(f)

    rows = []
    for load_str, trials in raw.items():
        load_kg = float(load_str)
        for trial_idx, trial_samples in enumerate(trials, start=1):
            for sample_idx, item in enumerate(trial_samples, start=1):
                rows.append(
                    {
                        "real_weight_kg": load_kg,
                        "trial": trial_idx,
                        "sample": sample_idx,
                        "estimated_fz_n": float(item["ext_wrench"][2]),
                    }
                )

    df = pd.DataFrame(rows)
    df = df.sort_values(["real_weight_kg", "trial", "sample"]).reset_index(drop=True)

    # display(df.head())
    # display(df.groupby("real_weight_kg")["estimated_fz_n"].agg(["count", "mean", "std"]))
    return df

# load train data
df = load_data("../data/train/load_test_data.json")


In [ ]:

"""设置绘图风格"""
# # 启用 Seaborn 的白色网格主题
# sns.set_theme(style="whitegrid")
# 设置字体类型
plt.rcParams.update(
    {
        # 设置中文字体（防止乱码，视系统环境而定，这里主要使用英文标签以确保通用性）；
        # 同时解决负号显示问题 (有些字体不支持负号，会显示方块)
        "axes.unicode_minus": False,
        # 设置字体为 Arial 或其他常见的无衬线字体
        # Linux (如 Ubuntu)，通常没有 Arial，但预装了开源字体 DejaVu Sans
        # 因此我们可以指定一个字体列表，Matplotlib 会按顺序查找可用字体并使用第一个找到的字体
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "DejaVu Sans", "Liberation Sans"],
    }
)
# 设置字体大小
# plt.rcParams.update(
#     {
#         "font.size": 14,
#         "axes.titlesize": 16,
#         "axes.labelsize": 14,
#         "xtick.labelsize": 12,
#         "ytick.labelsize": 12,
#         "legend.fontsize": 13,
#     }
# )


def style_axes(ax: plt.Axes) -> None:
    # 移除顶部和右侧的边框线（spines）
    sns.despine(ax=ax, top=True, right=True, left=False, bottom=False)
    # 设置剩余边框线的颜色和宽度
    # for side in ("left", "bottom"):
    #     ax.spines[side].set_color("black")
    #     ax.spines[side].set_linewidth(1.2)
    # 将网格线放在数据点下方
    ax.set_axisbelow(True)
    # 将水平网格线设置为虚线，颜色为浅灰色，适当调整透明度
    ax.grid(False)
    ax.yaxis.grid(True, linestyle="--", linewidth=0.8, color="0.7", alpha=0.8)
    ax.xaxis.grid(False)


In [ ]:
# 1. 散点图
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df,
    x="real_weight_kg",
    y="estimated_fz_n",
    s=45,
    alpha=0.75,
)
plt.xlabel("Real Weight (kg)")
plt.ylabel("Estimated Z-axis Force (N)")
plt.title("Scatter: Weight vs Estimated Z Force")
plt.tight_layout()
plt.show()

In [ ]:
# 2. 箱线图
order = sorted(df["real_weight_kg"].unique())
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=df,
    x="real_weight_kg",
    y="estimated_fz_n",
    order=order,
    width=0.6,
)
plt.xlabel("Real Weight (kg)")
plt.ylabel("Estimated Z-axis Force (N)")
plt.title("Box Plot: Estimated Z Force by Weight")
plt.tight_layout()
plt.show()

In [ ]:
# 3. 带误差线的柱状图
plt.figure(figsize=(8, 5))
sns.barplot(
    data=df,
    x="real_weight_kg",
    y="estimated_fz_n",
    order=sorted(df["real_weight_kg"].unique()),
    estimator="mean",
    errorbar="sd",
    capsize=0.15,
)
plt.xlabel("Real Weight (kg)")
plt.ylabel("Estimated Z-axis Force (N)")
plt.title("Bar Plot with SD Error Bars")
plt.tight_layout()
plt.show()

In [ ]:
# 4. 带标准差的折线图(趋势图)
plt.figure(figsize=(8, 5))
sns.lineplot(
    data=df,
    x="real_weight_kg",
    y="estimated_fz_n",
    estimator="mean",
    errorbar="sd",
    marker="o",
    linewidth=2.0,
)
plt.xlabel("Real Weight (kg)")
plt.ylabel("Estimated Z-axis Force (N)")
plt.title("Trend Line with SD Band")
plt.tight_layout()
# NOTE: use gcf before show, otherwise the saved figure will be blank
save_path = Path("data/trend_line_with_sd_band")
save_path.parent.mkdir(parents=True, exist_ok=True)
save_publication_figure(plt.gcf(), save_path, formats=["png", "pdf"])
plt.show()

In [ ]:
# 5. 散点 + 线性拟合线


def fit_data(df: pd.DataFrame):
    x = df["real_weight_kg"].to_numpy()
    y = df["estimated_fz_n"].to_numpy()
    slope, intercept = np.polyfit(x, y, 1)
    x_fit = np.linspace(x.min(), x.max(), 200)
    y_fit = slope * x_fit + intercept
    return x_fit, y_fit, slope, intercept


x_fit, y_fit, slope, intercept = fit_data(df)

"""load and fit test data"""
df_test = load_data("../data/load_test_data.json")
x_fit_test, y_fit_test, slope_test, intercept_test = fit_data(df_test)
"""********"""

fig = plt.figure(figsize=(8, 5))
style_axes(plt.gca())

sns.scatterplot(
    data=df_test,
    x="real_weight_kg",
    y="estimated_fz_n",
    s=35,
    alpha=0.6,
    label="Samples",
)
sns.lineplot(
    x=x_fit_test,
    y=y_fit_test,
    color="tab:orange",
    linewidth=2.2,
    label=f"Fit: y={slope_test:.3f}x+{intercept_test:.3f}",
)
sns.scatterplot(
    data=df, x="real_weight_kg", y="estimated_fz_n", s=35, alpha=0.6, label="Samples"
)
sns.lineplot(
    x=x_fit,
    y=y_fit,
    color="tab:red",
    linewidth=2.2,
    label=f"Fit: y={slope:.3f}x+{intercept:.3f}",
)
sns.lineplot(x=x_fit, y=x_fit, color="tab:blue", linewidth=2.2, label="y=x")
plt.xlabel("Real Weight (kg)")
plt.ylabel("Estimated Z-axis Force (N)")
plt.title("Scatter + Linear Fit")
plt.tight_layout()
plt.show()

print(f"Linear fit: estimated_fz_n = {slope:.6f} * real_weight_kg + {intercept:.6f}")

In [ ]:
# 6. 拟合误差分析
error_fit = x_fit - y_fit
print(error_fit.shape)
np.mean(error_fit), np.std(error_fit)

In [ ]:
def calc_error(x_true):
    return x_true - (slope * x_true + intercept)

def cali_pred(x_true, x_pred):
    return x_pred + calc_error(x_true)

In [ ]:
"""
2kg:
force_z_mean: 21.206355208847935
force_z_std: 0.20458717999235873
1kg:
force_z_mean: 11.385438447899995
force_z_std: 0.19076687236277354
0.5kg:
force_z_mean: 6.80334428281384
force_z_std: 0.1697620917796726
0kg:
force_z_mean: 1.7614717910938857
force_z_std: 0.17299301846592474
1.5kg:
force_z_mean: 16.780938798190455
force_z_std: 0.19072673451772781
"""